# 8. Gene-Query JEPA embedding analysis

Notebook **04 + 05**, but for a **JEPA checkpoint** (not MaskGIT).

This run trained on **every pairing row** (`--split false`, `n_train=148107`, no val). The embedding dump is the same: `--eval-split all`. There is no frozen-split hold-out for JEPA.

Cell analysis matches MaskGIT: **one row per unique LPS `cell_pairing_index`** (mean over pair-reuse), **69,531 × 768**. The raw dump still has 444,288 rows; §8.3 collapses it. Genes are the same **1,856** tokens as notebook 04/05.

| Piece | What you get |
|---|---|
| ckpt | `epoch=02.ckpt` (best `train/gene_gap_vs_copy_src`) |
| dump | already on sod2 under this run’s `embeddings/` (`eval_split=all`, 148107 cells) |
| cells | unique LPS: mean `z_hat` / `z_src` / `z_tgt` |
| genes | mean query embeddings in `varm` (90m / 6h / 10h) |

Re-dump only if you set `FORCE_DUMP = True` (needs **one GPU**; default `--gpu 7`).


## 8.1. Parameters

Paths for the 17 Aug full-atlas JEPA run. No split pickle: training and dump both used every cell.


In [2]:
import os
from pathlib import Path

WORKSPACE = Path("/home/stuke1/perturbgen")
REPO = WORKSPACE / "Perturbgen"
TOKENIZED = WORKSPACE / "T_perturb" / "tokenized_data" / "LPS_all_tps_2k"
SOD2 = Path("/mnt/sod2-project/csb4/stuke1/perturbgen")

GPU = 7  # physical device id passed as --gpu
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
os.environ["WANDB_MODE"] = "disabled"
# Do not set Agg here — plot cells need inline figures. Dump subprocess sets Agg itself.
os.environ.pop("MPLBACKEND", None)
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"
os.environ["NUMBA_CACHE_DIR"] = "/tmp/numba_cache"

RUN_DIR = (
    SOD2 / "T_perturb" / "res" / "jepa_gene_query_full_atlas"
    / "fzF_encL3_predL3_q128_mixed_lg1_lc0.1_contr0_vic1_0.04_bs16_ep5_lr0.0001_seed0_splitF_20260817_174952"
)
CKPT = RUN_DIR / "checkpoints" / "epoch=02.ckpt"
EMB_DIR = RUN_DIR / "embeddings"
CELL_H5AD = EMB_DIR / "jepa_cell_embeddings.h5ad"
GENE_H5AD = EMB_DIR / "jepa_gene_embeddings.h5ad"
FIG_DIR = EMB_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

EVAL_SPLIT = "all"  # matches training: every pairing row
BATCH_SIZE = 16
FORCE_DUMP = False

assert CKPT.is_file(), f"missing: {CKPT}"

print("CKPT:", CKPT)
print("RUN_DIR:", RUN_DIR)
print("CELL_H5AD:", CELL_H5AD)
print("GPU:", GPU)
print("eval_split:", EVAL_SPLIT, "(train was also split=false)")


CKPT: /mnt/sod2-project/csb4/stuke1/perturbgen/T_perturb/res/jepa_gene_query_full_atlas/fzF_encL3_predL3_q128_mixed_lg1_lc0.1_contr0_vic1_0.04_bs16_ep5_lr0.0001_seed0_splitF_20260817_174952/checkpoints/epoch=02.ckpt
RUN_DIR: /mnt/sod2-project/csb4/stuke1/perturbgen/T_perturb/res/jepa_gene_query_full_atlas/fzF_encL3_predL3_q128_mixed_lg1_lc0.1_contr0_vic1_0.04_bs16_ep5_lr0.0001_seed0_splitF_20260817_174952
CELL_H5AD: /mnt/sod2-project/csb4/stuke1/perturbgen/T_perturb/res/jepa_gene_query_full_atlas/fzF_encL3_predL3_q128_mixed_lg1_lc0.1_contr0_vic1_0.04_bs16_ep5_lr0.0001_seed0_splitF_20260817_174952/embeddings/jepa_cell_embeddings.h5ad
GPU: 7
eval_split: all
split_path (unused unless eval_split=test): /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/splits/stratified_cell_type_harmonized_seed42_90_10.pkl


## 8.2. Dump embeddings (skip if h5ad exists)

`train_gene_query_jepa.py --eval-ckpt` loads the Lightning checkpoint, runs `test` on **all** pairing indices (`--eval-split all`), then writes:

- `embeddings/gene_query_jepa_embeddings.pt`
- `embeddings/jepa_cell_embeddings.h5ad` — one row per **pairing row × target time** (pair reuse; ~444k)
- `embeddings/jepa_gene_embeddings.h5ad` — **1,856** genes in `var`; time-wise means in `varm`

This dump is already on disk (`n_eval_cells=148107`). §8.3 writes `jepa_cell_embeddings_unique.h5ad` for plotting.


In [2]:
import subprocess

cmd = [
    "python", "docs/examples/train_gene_query_jepa.py",
    "--eval-ckpt", str(CKPT),
    "--eval-split", "all",
    "--eval-force", "true" if FORCE_DUMP else "false",
    "--gpu", str(GPU),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", "2",
    "--tokenized", str(TOKENIZED),
]
print(" ".join(cmd))

if CELL_H5AD.is_file() and not FORCE_DUMP:
    print("Skip dump; already have", CELL_H5AD)
else:
    env = os.environ.copy()
    env.pop("CUDA_VISIBLE_DEVICES", None)
    env["PYTHONPATH"] = str(REPO)
    env["PYTHONUNBUFFERED"] = "1"
    subprocess.run(cmd, check=True, cwd=str(REPO), env=env)
    print("Done.")

assert CELL_H5AD.is_file(), CELL_H5AD
assert GENE_H5AD.is_file(), GENE_H5AD


python docs/examples/train_gene_query_jepa.py --eval-ckpt /mnt/sod2-project/csb4/stuke1/perturbgen/T_perturb/res/jepa_gene_query_full_atlas/fzF_encL3_predL3_q128_mixed_lg1_lc0.1_contr0_vic1_0.04_bs16_ep5_lr0.0001_seed0_splitF_20260817_174952/checkpoints/epoch=02.ckpt --eval-split all --eval-split-path /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/splits/stratified_cell_type_harmonized_seed42_80_10_10.pkl --eval-force false --gpu 7 --batch-size 16 --num-workers 2 --tokenized /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k
Skip dump; already have /mnt/sod2-project/csb4/stuke1/perturbgen/T_perturb/res/jepa_gene_query_full_atlas/fzF_encL3_predL3_q128_mixed_lg1_lc0.1_contr0_vic1_0.04_bs16_ep5_lr0.0001_seed0_splitF_20260817_174952/embeddings/jepa_cell_embeddings.h5ad


## 8.3. Cell embeddings (notebook 04 analogue)

The dump repeats each LPS cell once per rest partner. Collapse to **unique `cell_pairing_index`**, averaging `z_hat` / `z_src` / `z_tgt` (same idea as MaskGIT `mean_duplicates`). Each ID has one LPS time, so times are not mixed.

`X` and `obsm['z_hat_cell']` (also copied to `cls_embeddings`) are the predictor pool. Source / true-target pools stay in `z_src_cell` / `z_tgt_cell`. Neighbors use **PCA-50**, like notebook 04. Run the plot cells when you want figures.

In [3]:
import os
os.environ.pop("MPLBACKEND", None)

%matplotlib inline
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline", force=True)

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import Image, display

sc.settings.figdir = str(FIG_DIR)
sc.settings.set_figure_params(dpi=120, facecolor="white")

adata_raw = sc.read_h5ad(CELL_H5AD)
print("dump (pair reuse x 3 times):", adata_raw.n_obs, "x", adata_raw.n_vars)
print("obsm:", list(adata_raw.obsm.keys()))

pair_col = "cell_pairing_index"
if pair_col not in adata_raw.obs:
    raise KeyError(pair_col)
obs = adata_raw.obs.copy()
order = obs.drop_duplicates(pair_col)[pair_col]


def mean_by_pair(mat):
    df = pd.DataFrame(np.asarray(mat), index=obs[pair_col].to_numpy())
    return df.groupby(level=0).mean().reindex(order).to_numpy()


z_hat = mean_by_pair(adata_raw.obsm["z_hat_cell"])
z_src = mean_by_pair(adata_raw.obsm["z_src_cell"])
z_tgt = mean_by_pair(adata_raw.obsm["z_tgt_cell"])
obs_u = obs.drop_duplicates(pair_col).copy()
obs_u.index = pd.Index(obs_u[pair_col].astype(str).to_numpy(), name=None)

adata = ad.AnnData(X=z_hat.copy(), obs=obs_u)
adata.obsm["z_hat_cell"] = z_hat
adata.obsm["z_src_cell"] = z_src
adata.obsm["z_tgt_cell"] = z_tgt
adata.obsm["cls_embeddings"] = z_hat  # MaskGIT name, for a side-by-side recipe

UNIQUE_CELL_H5AD = EMB_DIR / "jepa_cell_embeddings_unique.h5ad"
adata.write_h5ad(UNIQUE_CELL_H5AD)

print("unique LPS cells (MaskGIT-style):", adata.n_obs, "x", adata.n_vars)
print("wrote", UNIQUE_CELL_H5AD)
print(adata.obs["time_after_LPS"].astype(str).value_counts())
print(adata.obs.head())
assert adata.n_obs == int(obs[pair_col].nunique())


dump (pair reuse x 3 times): 444288 x 768
obsm: ['z_hat_cell', 'z_src_cell', 'z_tgt_cell']
unique LPS cells (MaskGIT-style): 69531 x 768
wrote /mnt/sod2-project/csb4/stuke1/perturbgen/T_perturb/res/jepa_gene_query_full_atlas/fzF_encL3_predL3_q128_mixed_lg1_lc0.1_contr0_vic1_0.04_bs16_ep5_lr0.0001_seed0_splitF_20260817_174952/embeddings/jepa_cell_embeddings_unique.h5ad
time_after_LPS
6h_LPS     39185
10h_LPS    19713
90m_LPS    10633
Name: count, dtype: int64
        time_step time_after_LPS  cell_pairing_index cell_type_harmonized
79225           1        90m_LPS               79225               B cell
126492          1        90m_LPS              126492               B cell
70796           1        90m_LPS               70796               B cell
76225           1        90m_LPS               76225               B cell
70837           1        90m_LPS               70837               B cell


In [5]:
# Unique LPS cells only. PCA-50 then neighbors, same as notebook 04.
if "X_umap" not in adata.obsm:
    sc.pp.pca(adata, n_comps=50)
    sc.pp.neighbors(adata, use_rep="X_pca", n_neighbors=15)
    sc.tl.umap(adata)

from IPython.display import Image, display

sc.pl.umap(
    adata,
    color=[c for c in ("cell_type_harmonized", "time_after_LPS") if c in adata.obs],
    wspace=0.4,
    save="_jepa_cell_hat_unique.png",
    show=True,
)
display(Image(str(FIG_DIR / "umap_jepa_cell_hat_unique.png")))


Stack source / prediction / target on the **unique LPS** table so you can see whether `z_hat` sits nearer the target than a copy of the source.

In [6]:
parts = []
for name, key in (("src", "z_src_cell"), ("hat", "z_hat_cell"), ("tgt", "z_tgt_cell")):
    part = ad.AnnData(X=np.asarray(adata.obsm[key]), obs=adata.obs.copy())
    part.obs["embedding"] = name
    parts.append(part)
stacked = ad.concat(parts, index_unique="-")
sc.pp.pca(stacked, n_comps=50)
sc.pp.neighbors(stacked, use_rep="X_pca", n_neighbors=15)
sc.tl.umap(stacked)
sc.pl.umap(
    stacked,
    color=[c for c in ("embedding", "cell_type_harmonized", "time_after_LPS") if c in stacked.obs],
    wspace=0.4,
    save="_jepa_cell_src_hat_tgt_unique.png",
    show=True,
)
display(Image(str(FIG_DIR / "umap_jepa_cell_src_hat_tgt_unique.png")))


SystemError: CPUDispatcher(<function nn_descent at 0x7f066603a700>) returned a result with an exception set

Per-time cosine: `hat` vs `tgt` should beat `src` vs `tgt` if the predictor learned dynamics (same honesty idea as `gene_gap_vs_copy_src`, at cell-pool level).

In [ ]:
def row_cosine(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    a = a / np.clip(np.linalg.norm(a, axis=1, keepdims=True), 1e-8, None)
    b = b / np.clip(np.linalg.norm(b, axis=1, keepdims=True), 1e-8, None)
    return (a * b).sum(axis=1)

rows = []
time_col = "time_after_LPS" if "time_after_LPS" in adata.obs else "time_step"
for tp, sub in adata.obs.groupby(time_col, observed=True):
    idx = sub.index
    hat = row_cosine(adata[idx].obsm["z_hat_cell"], adata[idx].obsm["z_tgt_cell"])
    cpy = row_cosine(adata[idx].obsm["z_src_cell"], adata[idx].obsm["z_tgt_cell"])
    rows.append(
        {
            "time": tp,
            "n": len(idx),
            "cos_hat_tgt": float(np.mean(hat)),
            "cos_src_tgt": float(np.mean(cpy)),
            "gap_hat_minus_copy": float(np.mean(hat - cpy)),
        }
    )
cell_gap = pd.DataFrame(rows)
display(cell_gap)
cell_gap.to_csv(EMB_DIR / "cell_cosine_vs_copy.csv", index=False)


## 8.4. Gene programs (notebook 05 analogue)

Same **1,856** gene tokens as MaskGIT notebook 04/05. Vectors are **running means** over random queries (present genes only). `varm['90m_LPS']` etc. are **predicted** (`hat`) means, so the stack/Leiden recipe matches notebook 05.

These means were accumulated on the **full pairing dump** (reused LPS cells over-weighted). They are not re-estimated on the 69,531 unique rows. `src_*` exists only for genes that appeared as shared queries (present in rest and target).

In [ ]:
gdata = sc.read_h5ad(GENE_H5AD)
print("gene tokens (should match MaskGIT 1856):", gdata.n_vars)
print(gdata)
print("varm:", list(gdata.varm.keys()))
gdata.var.head()


In [ ]:
timepoints = ["90m_LPS", "6h_LPS", "10h_LPS"]
missing = [t for t in timepoints if t not in gdata.varm]
assert not missing, missing

min_queries = 20
keep = gdata.var["n_queries_hat"].to_numpy() >= min_queries
gkeep = gdata[:, keep]
symbols = gkeep.var["gene_symbol"].astype(str).to_numpy()

blocks, names, genes, tps = [], [], [], []
for tp in timepoints:
    blocks.append(np.asarray(gkeep.varm[tp]))
    names.extend([f"{s}_{tp}" for s in symbols])
    genes.extend(symbols)
    tps.extend([tp] * len(symbols))

gene_adata = ad.AnnData(
    X=np.vstack(blocks),
    obs=pd.DataFrame({"gene_symbol": genes, "timepoint": tps}, index=names),
)
zero = (gene_adata.X == 0).all(axis=1)
print(f"all-zero gene-time rows: {int(zero.sum())} / {gene_adata.n_obs}")
gene_adata = gene_adata[~zero].copy()

sc.pp.neighbors(gene_adata, use_rep="X", n_neighbors=15)
sc.tl.umap(gene_adata)
sc.tl.leiden(gene_adata, resolution=1.0, key_added="r1")
sc.pl.umap(gene_adata, color=["timepoint", "r1"], wspace=0.4, save="_jepa_gene_hat.png", show=True)
display(Image(str(FIG_DIR / "umap_jepa_gene_hat.png")))
gene_adata.write_h5ad(EMB_DIR / "jepa_gene_programs_hat.h5ad")
print(gene_adata)


Same stack for **true target** gene means (`tgt_*`). If Leiden structure is similar to `hat`, the predictor is preserving program geometry, not inventing a private space.

In [ ]:
tgt_keys = [f"tgt_{t}" for t in timepoints]
if all(k in gkeep.varm for k in tgt_keys):
    blocks, names, genes, tps = [], [], [], []
    for tp, key in zip(timepoints, tgt_keys):
        blocks.append(np.asarray(gkeep.varm[key]))
        names.extend([f"{s}_{tp}" for s in symbols])
        genes.extend(symbols)
        tps.extend([tp] * len(symbols))
    tgt_adata = ad.AnnData(
        X=np.vstack(blocks),
        obs=pd.DataFrame({"gene_symbol": genes, "timepoint": tps}, index=names),
    )
    tgt_adata = tgt_adata[~(tgt_adata.X == 0).all(axis=1)].copy()
    sc.pp.neighbors(tgt_adata, use_rep="X", n_neighbors=15)
    sc.tl.umap(tgt_adata)
    sc.tl.leiden(tgt_adata, resolution=1.0, key_added="r1")
    sc.pl.umap(tgt_adata, color=["timepoint", "r1"], wspace=0.4, save="_jepa_gene_tgt.png", show=True)
    display(Image(str(FIG_DIR / "umap_jepa_gene_tgt.png")))
else:
    print("no tgt_* varm keys; skip")


Gene-level honesty: cosine(`hat`, `tgt`) minus cosine(`src`, `tgt`) on genes that have a source mean (shared queries).

In [ ]:
gene_rows = []
for tp in timepoints:
    hat = np.asarray(gkeep.varm[f"hat_{tp}"] if f"hat_{tp}" in gkeep.varm else gkeep.varm[tp])
    tgt = np.asarray(gkeep.varm[f"tgt_{tp}"])
    src_key = f"src_{tp}"
    hat_c = row_cosine(hat, tgt)
    rec = {"time": tp, "cos_hat_tgt": float(np.mean(hat_c))}
    if src_key in gkeep.varm:
        src = np.asarray(gkeep.varm[src_key])
        both = ~((src == 0).all(axis=1) | (tgt == 0).all(axis=1))
        if both.any():
            cpy = row_cosine(src[both], tgt[both])
            rec["cos_src_tgt"] = float(np.mean(cpy))
            rec["gap_hat_minus_copy"] = float(np.mean(row_cosine(hat[both], tgt[both]) - cpy))
            rec["n_shared"] = int(both.sum())
    gene_rows.append(rec)
gene_gap = pd.DataFrame(gene_rows)
display(gene_gap)
gene_gap.to_csv(EMB_DIR / "gene_cosine_vs_copy.csv", index=False)


Optional Enrichr on Leiden clusters (needs `gseapy`). Skip if the import fails.

In [ ]:
try:
    import gseapy as gp
except ImportError:
    print("gseapy not installed; skip Enrichr")
else:
    results = {}
    for cluster in sorted(gene_adata.obs["r1"].unique(), key=str):
        genes = (
            gene_adata.obs.loc[gene_adata.obs["r1"] == cluster, "gene_symbol"]
            .astype(str)
            .unique()
            .tolist()
        )
        genes = [g for g in genes if g and not g.startswith("ENSG")]
        if len(genes) < 8:
            continue
        try:
            enr = gp.enrichr(
                gene_list=genes,
                gene_sets=["GO_Biological_Process_2023"],
                organism="human",
                outdir=None,
            )
            results[str(cluster)] = enr.results.head(10)
        except Exception as exc:
            print("cluster", cluster, "failed:", exc)
    if results:
        top = next(iter(results.values()))
        display(top)


## 8.5. Cell-type probe (notebook 04 analogue)

Same 5-fold logistic regression on **PCA-50** of unique LPS `z_hat`. MaskGIT number below is the **16 Aug atlas** (old 80/10/10 MaskGIT), not the new 90/10 extract. Time probe is included so you can compare to the mixed-time UMAP.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import silhouette_score

if "X_pca" not in adata.obsm:
    sc.pp.pca(adata, n_comps=50)

X = np.asarray(adata.obsm["X_pca"])


def probe(X, y, name):
    y = np.asarray(y).astype(str)
    mask = np.array([lab not in ("nan", "None", "") for lab in y])
    Xp, yp = X[mask], y[mask]
    print(f"{name} n: {Xp.shape[0]} classes: {len(np.unique(yp))}")
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000),
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    scores = cross_val_score(clf, Xp, yp, cv=cv, scoring="accuracy", n_jobs=1)
    print(f"5-fold {name} accuracy: {scores.mean():.3f} ± {scores.std():.3f}")
    print("folds:", np.round(scores, 3))
    rng = np.random.default_rng(0)
    n_sil = min(10_000, Xp.shape[0])
    idx = rng.choice(Xp.shape[0], size=n_sil, replace=False)
    sil = silhouette_score(Xp[idx], yp[idx], metric="euclidean")
    print(f"silhouette ({name}, {n_sil} cells): {sil:.3f}")
    return scores


probe(X, adata.obs["cell_type_harmonized"], "cell type")
print("MaskGIT 16 Aug atlas (old split) cell-type accuracy: 0.988 ± 0.001")
probe(X, adata.obs["time_after_LPS"], "time")
print("time majority baseline (always 6h):", f"{39185 / 69531:.3f}")
